In [1]:
import uuid
import logging
from base64 import b64encode
from dotenv import load_dotenv

logging.basicConfig(level=logging.INFO)
logging.getLogger(__name__).setLevel(logging.DEBUG)
logging.getLogger("httpx").setLevel(logging.WARNING)
logger = logging.getLogger(__name__)
load_dotenv()

True

## Evaluation pipeline

### Prepare data

In [2]:
from evals import Dataset

dataset_name = "THRD-2021-163881"
dc = Dataset(dataset_name)
data = dc.load_dataset()

if not data or "sessions" not in data:
    print("Ingen sessions funnet")
    exit()

dc.assign_session_attachments()

total_attachments = sum(len(s.get("attachments", [])) for s in data["sessions"])
logger.info(f"Done — {len(data['sessions'])} sessions, {total_attachments} attachments assigned in total")

INFO:evals.dataset:Found 120 files under datasets/THRD-2021-163881/01_data/
INFO:evals.dataset:Session Prosjekt-initialisering | – → 2020-03-15 | 18 candidates, 18 new
INFO:evals.dataset:Session Forbehold om heving | 2020-03-15 → 2020-05-20 | 7 candidates, 7 new
INFO:evals.dataset:Session Formell heving | 2020-05-20 → 2021-06-25 | 7 candidates, 7 new
INFO:evals.dataset:Session Stevning og tilsvar | 2021-06-25 → 2022-01-15 | 5 candidates, 5 new
INFO:evals.dataset:Session Forberedelse rettsmekling | 2022-01-15 → 2022-03-10 | 2 candidates, 2 new
INFO:evals.dataset:Session Forliksavtale | 2022-03-10 → 2022-04-05 | 4 candidates, 4 new
INFO:evals.dataset:Session Forliksbrudd | 2022-04-05 → 2023-04-20 | 24 candidates, 24 new
INFO:evals.dataset:Session Gjenopptakelse | 2023-04-20 → 2023-07-10 | 7 candidates, 7 new
INFO:evals.dataset:Session Prosesskriv og sluttinnlegg | 2023-07-10 → 2025-06-05 | 23 candidates, 23 new
INFO:evals.dataset:Session Dom | 2025-06-05 → 2025-07-20 | 20 candidates, 20 

### GATHER RESULTS

In [ ]:
from agent.agent import Agent
from models import AskAgentRequest, AttachmentModel
from agent.utils import PROMPT
from agent.tools import TOOLS
import os
from langgraph.checkpoint.postgres.aio import AsyncPostgresSaver
from psycopg_pool import AsyncConnectionPool

class CollectAgentResult:
    def __init__(self, data: dict, llm_model: str = "google_gemini-2.5-pro"):
        self.data = data
        self.llm_model = llm_model
        self.custom_agent : bool = True
        self.dataclass = Dataset(name=data.get("dataset_name", "default_dataset"))

    async def init_agent(self, use_factsheet : bool = True, save_to_storage: bool = True, embed_to_vectorstore: bool = True):
        connection_string = os.getenv("SUPABASE_DB_URL")
        pool = AsyncConnectionPool(conninfo=connection_string, open=False)
        await pool.open()
        checkpointer = AsyncPostgresSaver(pool)
        agent = Agent(
            use_factsheet=use_factsheet,
            save_to_storage=save_to_storage,
            embed_to_vectorstore=embed_to_vectorstore,
            tools=TOOLS,
            prompt=PROMPT,
            checkpointer=checkpointer,
        )
        logger.info("Agent initialized with AsyncPostgresSaver checkpointer")
        return agent

    def file_type_map(self, blob_path: str):
        mapping = {
            "txt": "text/plain",
            "pdf": "application/pdf",
            "docx": "application/vnd.openxmlformats-officedocument.wordprocessingml.document",
            "xlsx": "application/vnd.openxmlformats-officedocument.spreadsheetml.sheet",
            "csv": "text/csv",
            "md": "text/markdown",
            "eml": "message/rfc822",
        }
        return mapping.get(blob_path.split(".")[-1], None)

    def parse_attachments(self, attachment_path: str, query_id: str, session_id: str, user_id: str):
        bytes_content = self.dataclass.bucket.blob(attachment_path).download_as_bytes()
        content = b64encode(bytes_content).decode("utf-8")
        file_id = str(uuid.uuid4())
        return AttachmentModel(
            filename=attachment_path,
            file_type=self.file_type_map(attachment_path),
            content=content,
            file_id=file_id,
            path=f"{user_id}/{session_id}/{file_id}.{attachment_path.split('.')[-1]}",
            size=len(content),
            query_id=query_id,
        )

    async def run_conv(self, conv, agent_class, project_id, session_id, query_id, user_id):
        input_obj = AskAgentRequest(
            question=conv.get("input"),
            session_id=session_id,
            llm_model=self.llm_model,
            query_id=query_id,
            project_id=project_id,
            attachments=[],
        )
        answer = "No content"
        async for response in agent_class.stream_response(query=input_obj, user_id=user_id):
            if response.get("type") == "ai":
                answer = response.get("data", {}).get("token_stream", "No content")
        conv["model_response"] = answer

    async def run_agent(self,embed_to_vectorstore: bool = True, save_to_storage: bool = True):
        if self.custom_agent:
            use_factsheet = True
        else:
            use_factsheet = False
        agent_class = await self.init_agent(use_factsheet=use_factsheet, save_to_storage=save_to_storage, embed_to_vectorstore=embed_to_vectorstore)
        logger.info("=========== STARTING EVALUATION ===========")
        logger.info(
            f'Dataset: {self.data.get("dataset_name")} | '
            f'Sessions: {len(self.data.get("sessions", []))} | '
            f'Project: {self.data.get("project_id")} | '
            f'User: {self.data.get("user_id")}'
        )
        self.data["llm_model"] = self.llm_model
        self.data["custom_agent"] = self.custom_agent

        for idx, session in enumerate(self.data.get("sessions", [])):
            logger.info(
                f"Session {idx} | {session.get('date')} | "
                f"{session.get('session_name')} | "
                f"{len(session.get('attachments', []))} attachments"
            )
            query_id = str(uuid.uuid4())
            attachments = [
                self.parse_attachments(
                    attachment_path=att,
                    query_id=query_id,
                    session_id=session.get("session_id", "unknown_session"),
                    user_id=self.data.get("user_id", "unknown_user"),
                )
                for att in session.get("attachments", [])
            ]

            input_obj = AskAgentRequest(
                question=session.get("init_query"),
                session_id=session.get("session_id"),
                llm_model=self.llm_model,
                query_id=query_id,
                project_id=self.data.get("project_id"),
                attachments=attachments,
            )

            if idx == 0:
                async for response in agent_class.initialize_project(
                    query=input_obj, user_id=self.data.get("user_id")
                ):
                    logger.debug(f"Init response: {response}")
            else:
                async for response in agent_class.update_project(
                    query=input_obj, user_id=self.data.get("user_id")
                ):
                    logger.debug(f"Update response: {response}")

            for conv in session["conversation"]:
                await self.run_conv(
                    conv=conv,
                    agent_class=agent_class,
                    project_id=self.data.get("project_id"),
                    session_id=session.get("session_id"),
                    query_id=query_id,
                    user_id=self.data.get("user_id"),
                )

INFO:pikepdf._core:pikepdf C++ to Python logger bridge initialized
/Users/sigvardbratlie/Documents/Projects/master-thesis/.venv/lib/python3.13/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "output_schema" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]
/Users/sigvardbratlie/Documents/Projects/master-thesis/.venv/lib/python3.13/site-packages/langchain_tavily/tavily_research.py:97: UserWarning: Field name "stream" in "TavilyResearch" shadows an attribute in parent "BaseTool"
  class TavilyResearch(BaseTool):  # type: ignore[override, override]


In [ ]:
car = CollectAgentResult(data, llm_model="google_gemini-2.5-flash")
await car.run_agent(embed_to_vectorstore=False, save_to_storage=False)

INFO:__main__:Agent initialized with AsyncPostgresSaver checkpointer
INFO:__main__:=========== STARTING EVALUATION ===========
INFO:__main__:Dataset: THRD-2021-163881 | Sessions: 10 | Project: ee9ee007-92f7-4fdc-ba5a-d7cb55694241 | User: 53d63d18-cfa1-416e-96e8-770c8f66507b
INFO:__main__:Session 0 | 2020-03-15 | Prosjekt-initialisering | 18 attachments
DEBUG:__main__:Init response: {'type': 'status', 'phase': ['parse-documents'], 'status': 'starting', 'data': {'attachments': 18}, 'timestamp': '2026-02-24T13:35:47.396814', 'query_id': 'ab696fe5-3093-4079-85a8-7dae802cfa3c'}
DEBUG:__main__:Init response: {'type': 'status', 'phase': ['parse_doc'], 'status': 'starting', 'data': {'filename': 'datasets/THRD-2021-163881/01_data/1994-08-15_37_byggetillatelse_1994_hovedbygg.txt', 'file_id': '97aece08-aa2a-4284-8ded-8bd14e991b98', 'progress': 0, 'total': 18}, 'timestamp': '2026-02-24T13:35:47.397225', 'query_id': 'ab696fe5-3093-4079-85a8-7dae802cfa3c'}
DEBUG:__main__:Init response: {'type': 'sta

CancelledError: 

INFO:agent.context_manager:==== ATTACHMENT ELEMENT DEBUG == 
{'description': 'An email from Emma Hansen, a lawyer, to Stavanger Kommune requesting copies of building permits, approved drawings for additions/extensions, and completion certificates for Fjellveien 42A, covering the period from 2010 to 2020.', 'significance': 'medium', 'party_roles': ['Emma Hansen', 'Stavanger Kommune'], 'deadlines': [], 'damages': [], 'claims': [], 'file_id': 'cd6f5bcc-40f3-4cf4-9074-e7798371bfdc', 'key_provisions': [], 'file_date': '2020-04-15', 'category': 'correspondence'}
 
 ==== END OF ELEMENT DEBUG ====
INFO:agent.context_manager:Attachment #1: Successfully matched extracted file_id=cd6f5bcc-40f3-4cf4-9074-e7798371bfdc to original
INFO:agent.context_manager:Processing attachment #1: input_file_id=cd6f5bcc-40f3-4cf4-9074-e7798371bfdc, extracted_file_id=cd6f5bcc-40f3-4cf4-9074-e7798371bfdc
INFO:agent.context_manager:  -> Attachment #1 processed successfully with file_id=cd6f5bcc-40f3-4cf4-9074-e779837

In [9]:
dc.save_results(car.data)

INFO:evals.dataset:Results saved to datasets/THRD-2021-163881/04_results/google_gemini-2.5-flash_2026-02-24_13-41-07.json


### EVALUATE RESULTS

In [7]:
import deepeval

In [11]:
results = dc.load_results()
result = results.get("datasets/THRD-2021-163881/04_results/google_gemini-2.5-flash_2026-02-24_13-41-07.json")
session1 = result["sessions"][0]

In [12]:
session1

{'session': 0,
 'date': '2020-03-15',
 'session_id': '3eb54a6b-ca3d-4d21-9ea1-b48f4e0228e1',
 'session_name': 'Prosjekt-initialisering',
 'init_query': 'Jeg er advokat og representerer kjøperparet Anders og Berit Kristiansen i en \neiendomskjøpssak. De kjøpte en eiendom på Fjellveien 42A i Stavanger kommune \nden 1. juni 2019, med overtakelse 1. august 2019. \n            \n            Vi er nå i mars 2020 og det har dukket opp flere problemer med eiendommen. \n            Jeg trenger din hjelp til å organisere saksinnholdet, identifisere de juridiske \n            problemstillingene, og vurdere mulige tiltak.\n',
 'conversation': [{'input': 'Gi meg en kort og konsis oppsummering av sakens faktiske bakgrunn og utvikling så langt, basert på dokumentene jeg har lastet opp. \nFokuser på de viktigste hendelsene og problemstillingene. Hva er kjernen i saken?',
   'answer': "Anders og Berit Kristiansen kjøpte en eiendom på Fjellveien 42A i Stavanger kommune den 1. juni 2019, med overtakelse 